In [0]:
%run ../../config/utils

In [0]:
import pandas as pd
import datetime
import yaml
import sys
from pyspark.sql import SparkSession, functions as sqlf
from pyspark.sql.functions import date_add
from pyspark.sql.window import Window

In [0]:
with open('./config/config.yml', "r") as f:
    cfg = yaml.load(f, Loader=yaml.FullLoader)

p = cfg["params"]

today = datetime.datetime.today()
recent_saturday = today - datetime.timedelta(days=(today.weekday() + 2) % 7)
recent_saturday_str = recent_saturday.strftime('%Y-%m-%d')
score_date = recent_saturday - datetime.timedelta(weeks=2)
two_wks_back_saturday_str = score_date.strftime('%Y-%m-%d')

In [0]:
try:
    cube = (
        spark.table(fs_merge)
        .filter(sqlf.col("FISCAL_WEEK_END") == two_wks_back_saturday_str)
        .filter(sqlf.col("MBRSHP_STAT_CD") == p["membership_stat"])
        .filter(sqlf.col("LATEST_MBRSHP_EXP_DT") >= sqlf.current_date() - 95)
        .filter(sqlf.col("LATEST_MFI_TIER") > 0)
        .filter(sqlf.col("TEAM_MBR_IND") != 'Y')
    )
    print("cube data loaded")
except Exception as e:
    print("error loading cube data", e)
    sys.exit(1)

In [0]:
#creating extra variable for last 60 day active member filter
from pyspark.sql.functions import date_add
out_of_time_data = cube.withColumn('target_strt_dt_1', date_add(sqlf.col("FISCAL_WEEK_END"), 1))\
                    .withColumn('target_end_dt_2', date_add(sqlf.col("FISCAL_WEEK_END"), 14))\
                    .withColumn('last_60days', date_add(sqlf.col("FISCAL_WEEK_END"), -60+1))
out_of_time_data.select('FISCAL_WEEK_END', 'target_strt_dt_1', 'target_end_dt_2','last_60days').distinct().show()

In [0]:
#Reading ETL intermdiate files to create last 60 days trip column based on purchase history
purch_header = spark.table(silver_transaction_fiscal_header).filter(sqlf.col('PURCH_DT') >= '2022-01-01')\
				.filter(sqlf.col('SALES_CHANNEL_ID').isin([10, 30, 60]))\
				.select('PURCH_HDR_ID','SALES_CHANNEL_ID')


purch_detail = spark.table(silver_transaction_fiscal_detail).filter(sqlf.col('PURCH_DT') >= '2022-01-01')\
				.filter(sqlf.col('SALES_CTGRY_CD').isin(['03','01']))\
				.filter(sqlf.col('RETURN_IND')=='N')\
				.select('MBRSHP_SID','PURCH_DT','PURCH_HDR_ID','DISCOUNT_TYPE_CD','ARTICLE_NBR','EXTENDED_PRC_AMT','SALES_CTGRY_CD','MCH3_CD','MCH2_CD','MCH2_DESC','MCH1_CD','AH3_DESC','AH4_DESC')

purch = purch_detail.join(purch_header, ['PURCH_HDR_ID'], how='left')

purch_payment = spark.table(silver_transaction_fiscal_payment).filter(sqlf.col('PURCH_DT') >= '2022-01-01')\
				.select('MBRSHP_SID','PURCH_HDR_ID','PURCH_DT','TENDER_TYPE_CD','SALES_PYMT_AMT')

In [0]:
# Creating last 60 days trip, no. of trip and spend column at member sid level
sales_data = out_of_time_data.select('MBRSHP_SID','FISCAL_WEEK_END','target_strt_dt_1','target_end_dt_2','last_60days').join(purch,'MBRSHP_SID','left').fillna(0,subset=['EXTENDED_PRC_AMT'])


sales_data_v2 = sales_data.groupby('MBRSHP_SID','FISCAL_WEEK_END').agg(
                                                       
                                                       #Target sales
                                                       sqlf.sum(sqlf.when((sqlf.col('PURCH_DT').between(sqlf.col('target_strt_dt_1'), sqlf.col('target_end_dt_2'))) & (sqlf.col('SALES_CTGRY_CD') == '03') & (sqlf.col('MCH3_CD').isin('100000000','200000000','300000000','400000000')) & (sqlf.col('DISCOUNT_TYPE_CD').isNull()), sqlf.col('EXTENDED_PRC_AMT'))).alias('target_1-2_spend'),
                                                       # Trips
                                                       sqlf.countDistinct(sqlf.when((sqlf.col('PURCH_DT').between(sqlf.col('target_strt_dt_1'), sqlf.col('target_end_dt_2'))) & (sqlf.col('SALES_CTGRY_CD') == '03') & (sqlf.col('MCH3_CD').isin('100000000','200000000','300000000','400000000')) & (sqlf.col('DISCOUNT_TYPE_CD').isNull()), sqlf.col('PURCH_HDR_ID'))).alias('target_1-2_trips'),
                                                       sqlf.countDistinct(sqlf.when((sqlf.col('PURCH_DT').between(sqlf.col('last_60days'), sqlf.col('FISCAL_WEEK_END'))) & (sqlf.col('SALES_CTGRY_CD') == '03') & (sqlf.col('MCH3_CD').isin('100000000','200000000','300000000','400000000')) & (sqlf.col('EXTENDED_PRC_AMT') >0) & (sqlf.col('DISCOUNT_TYPE_CD').isNull()), sqlf.col('PURCH_HDR_ID'))).alias('last_60days_trips'),

)

print("Target data prepared")

In [0]:
## Creating max spend column at member level 

sales_data_transactions_2wks = sales_data.filter(sqlf.col('PURCH_DT').between(sqlf.col('target_strt_dt_1'), sqlf.col('target_end_dt_2'))).groupby('MBRSHP_SID','FISCAL_WEEK_END','PURCH_HDR_ID').agg(
                                                       
                                                       #Target
                                                       sqlf.sum(sqlf.when((sqlf.col('PURCH_DT').between(sqlf.col('target_strt_dt_1'), sqlf.col('target_end_dt_2'))) & (sqlf.col('SALES_CTGRY_CD') == '03') & 
                                                                          (sqlf.col('MCH3_CD').isin('100000000','200000000','300000000','400000000')) & (sqlf.col('DISCOUNT_TYPE_CD').isNull()), sqlf.col('EXTENDED_PRC_AMT'))).alias('target_1-2_spend_trans'),)
                                                        

sales_data_transactions_2wks_max = sales_data_transactions_2wks.groupby('MBRSHP_SID','FISCAL_WEEK_END').agg(
                                                       
                                                       #Target
                                                       sqlf.max('target_1-2_spend_trans').alias('target_1-2_spend_trans_max'),)
                                                        
print("Transaction data prepared")

In [0]:
# Joining above tables with key metrics into one by member sid and fiscal week end date
out_of_time_data_v2 = out_of_time_data.select('MBRSHP_SID','FISCAL_WEEK_END').join(sales_data_v2.select('MBRSHP_SID','FISCAL_WEEK_END','target_1-2_spend','target_1-2_trips','last_60days_trips'),['MBRSHP_SID','FISCAL_WEEK_END'],'left').fillna(0,subset=['target_1-2_spend','target_1-2_trips'])
out_of_time_data_v2 = out_of_time_data_v2.join(sales_data_transactions_2wks_max, ['MBRSHP_SID','FISCAL_WEEK_END'], 'left').fillna(0,subset=['target_1-2_spend_trans_max'])

In [0]:
#Converting the spark dataframe in pandas dataframe
out_of_time_data_v2_pd = out_of_time_data_v2.toPandas()

trips = spark.table(digital_trips_scores).filter(sqlf.col('FISCAL_WEEK_END')==two_wks_back_saturday_str)
sales = spark.table(digital_sales_scores).filter(sqlf.col('FISCAL_WEEK_END')==two_wks_back_saturday_str)

trips_df = trips.toPandas()
sales_df = sales.toPandas()

score_data = trips_df.merge(sales_df, left_on = ["MBRSHP_SID",'FISCAL_WEEK_END',"LATEST_MBRSHP_NBR"], right_on =["MBRSHP_SID",'FISCAL_WEEK_END',"LATEST_MBRSHP_NBR"],how='inner')

In [0]:
#Joining the score file with the dataset to get members for which we already have scores
final_score_pd = score_data.merge(out_of_time_data_v2_pd[["MBRSHP_SID"]],left_on=['MBRSHP_SID'], right_on=['MBRSHP_SID'],how='inner')

final_score_pd["rank_sales"] = final_score_pd["sales_score"].rank(method="first", ascending=False)
final_score_pd["sales_decile"] = pd.qcut(final_score_pd["rank_sales"], q=10, labels=range(1, 11))
final_score_pd["rank_trip"] = final_score_pd["trips_score"].rank(method="first", ascending=False)
final_score_pd["trip_decile"] = pd.qcut(final_score_pd["rank_trip"], q=10, labels=range(1, 11))

out_of_time_data_v2_pd.columns = out_of_time_data_v2_pd.columns.str.upper()

out_of_time_data_v2_pd['Shoppers'] = out_of_time_data_v2_pd['TARGET_1-2_TRIPS'].apply(lambda x: 1 if x >=1 else 0)

crosstab = final_score_pd.merge(out_of_time_data_v2_pd[["MBRSHP_SID", "TARGET_1-2_SPEND","TARGET_1-2_TRIPS","TARGET_1-2_SPEND_TRANS_MAX",'Shoppers']],left_on=['MBRSHP_SID'], right_on=['MBRSHP_SID'],how='inner')

In [0]:
#Avg Spend
avg_spend = crosstab.pivot_table(columns = 'sales_decile',
         index = 'trip_decile',
         values = 'TARGET_1-2_SPEND',
        aggfunc = 'mean')

print("Avg spend prepared")



#Avg Trips
avg_trips = crosstab.pivot_table(columns = 'sales_decile',
         index = 'trip_decile',
         values = 'TARGET_1-2_TRIPS',
        aggfunc = 'mean')


print("Avg trips prepared")


#Average Max Basket Size
avg_max_basket_size = crosstab.pivot_table(columns = 'sales_decile',
         index = 'trip_decile',
         values = 'TARGET_1-2_SPEND_TRANS_MAX',
        aggfunc = 'mean')


print("Average Max Basket Size prepared")


#Shoopers
per_of_shoppers = (crosstab.pivot_table(columns = 'sales_decile',
         index = 'trip_decile',
         values = 'Shoppers',
        aggfunc = 'mean'))*100


print("Per_of_shoppers prepared")

In [0]:
pip install xlsxwriter

In [0]:
import os
Decile_path = os.path.join(
    '.',  
    f"validation_analysis_{two_wks_back_saturday_str}.xlsx"
)

Volume_path = os.path.join(
    '/Volumes/datascience_ea_dev/pe/helpers/digital_propensity/scores_validation/',  
    f"validation_analysis_{two_wks_back_saturday_str}.xlsx"
)


# Write Excel analysis to the temporary file
with pd.ExcelWriter(Decile_path, engine='xlsxwriter') as writer:
    avg_spend.to_excel(writer, sheet_name='Avg_Spend', index=False)
    avg_trips.to_excel(writer, sheet_name='Avg_Trips')
    avg_max_basket_size.to_excel(writer, sheet_name='Avg_Max_Bucket_Size')
    per_of_shoppers.to_excel(writer, sheet_name='Percent_of_Shoppers')

In [0]:
from shutil import move

move(Decile_path, Volume_path)